In [0]:
# Install required libraries untuk extract text dari PDF dan DOCX
%pip install PyPDF2 python-docx scikit-learn --quiet
dbutils.library.restartPython()

In [0]:
# Cek file apa saja yang ada di Volumes
import os

# Path ke volumes (sesuaikan dengan struktur Anda)
base_path = "/Volumes/workspace/bronze"

print("🔍 Mencari files di Volumes...\n")
print("="*60)

try:
    # List all items in the path
    items = os.listdir(base_path)
    
    if items:
        print(f"✅ Ditemukan {len(items)} item di {base_path}:\n")
        
        for item in items:
            full_path = os.path.join(base_path, item)
            
            # Check if it's a directory (volume) or file
            if os.path.isdir(full_path):
                print(f"📁 FOLDER: {item}")
                
                # List files inside the folder
                try:
                    files_in_folder = os.listdir(full_path)
                    for file in files_in_folder:
                        file_path = os.path.join(full_path, file)
                        file_size = os.path.getsize(file_path) / 1024  # KB
                        print(f"   📄 {file} ({file_size:.1f} KB)")
                        print(f"      Path: {file_path}")
                except:
                    pass
            else:
                file_size = os.path.getsize(full_path) / 1024  # KB
                print(f"📄 FILE: {item} ({file_size:.1f} KB)")
                print(f"   Path: {full_path}")
        
        print("\n" + "="*60)
        print("✅ Copy path di atas untuk digunakan di Cell 5!")
    else:
        print("⚠️  Folder kosong")
        
except FileNotFoundError:
    print(f"❌ Path tidak ditemukan: {base_path}")
    print("\n💡 Coba path alternatif:")
    
    alternative_paths = [
        "/Volumes/workspace/bronze",
        "/Volumes/workspace/default",
        "/dbfs/FileStore/"
    ]
    
    for alt_path in alternative_paths:
        print(f"   - {alt_path}")
        try:
            if os.path.exists(alt_path):
                print(f"     ✅ Path ini exist!")
        except:
            pass
            
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n💡 Alternatif: Gunakan Catalog UI untuk cek path")
    print("   1. Klik 'Catalog' di sidebar kiri")
    print("   2. Navigate: workspace > bronze > [volume_name]")
    print("   3. Klik file > Copy path")

In [0]:
from PyPDF2 import PdfReader
import os

def extract_text_from_pdf(file_path):
    """
    Extract text dari file PDF
    
    Args:
        file_path: Path ke file PDF (bisa /Volumes atau /dbfs)
    
    Returns:
        String berisi semua text dari PDF
    """
    try:
        reader = PdfReader(file_path)
        text = ""
        
        for page_num, page in enumerate(reader.pages, 1):
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        
        print(f"✅ Berhasil extract {len(reader.pages)} halaman")
        print(f"📊 Total karakter: {len(text):,}")
        return text.strip()
    
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Test function (uncomment jika mau test dengan file dummy)
# Contoh: text = extract_text_from_pdf("/Volumes/catalog/schema/volume/document.pdf")

In [0]:
from docx import Document

def extract_text_from_docx(file_path):
    """
    Extract text dari file DOCX (Word)
    
    Args:
        file_path: Path ke file DOCX
    
    Returns:
        String berisi semua text dari DOCX
    """
    try:
        doc = Document(file_path)
        text = ""
        
        # Extract dari paragraphs
        for para in doc.paragraphs:
            if para.text.strip():
                text += para.text + "\n"
        
        # Extract dari tables
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    if cell.text.strip():
                        text += cell.text + " "
            text += "\n"
        
        print(f"✅ Berhasil extract {len(doc.paragraphs)} paragraphs dan {len(doc.tables)} tables")
        print(f"📊 Total karakter: {len(text):,}")
        return text.strip()
    
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Test function
# Contoh: text = extract_text_from_docx("/Volumes/catalog/schema/volume/document.docx")

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def calculate_similarity(text1, text2, show_details=True):
    """
    Hitung similarity antara 2 dokumen menggunakan TF-IDF + Cosine Similarity
    
    Args:
        text1: Text dari dokumen pertama
        text2: Text dari dokumen kedua
        show_details: Tampilkan detail analisis (default: True)
    
    Returns:
        Dictionary berisi similarity score dan detail
    """
    if not text1 or not text2:
        return {"error": "Text tidak boleh kosong"}
    
    # Vectorize text menggunakan TF-IDF
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words='english',  # Filter common words
        max_features=5000  # Max 5000 most important words
    )
    
    try:
        tfidf_matrix = vectorizer.fit_transform([text1, text2])
        
        # Hitung cosine similarity
        similarity_score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        
        # Get vocabulary info
        feature_names = vectorizer.get_feature_names_out()
        
        # Get top words from each document
        doc1_vector = tfidf_matrix[0].toarray()[0]
        doc2_vector = tfidf_matrix[1].toarray()[0]
        
        # Top 10 keywords from doc1
        top_indices_doc1 = doc1_vector.argsort()[-10:][::-1]
        top_keywords_doc1 = [feature_names[i] for i in top_indices_doc1 if doc1_vector[i] > 0]
        
        # Top 10 keywords from doc2
        top_indices_doc2 = doc2_vector.argsort()[-10:][::-1]
        top_keywords_doc2 = [feature_names[i] for i in top_indices_doc2 if doc2_vector[i] > 0]
        
        result = {
            "similarity_score": similarity_score,
            "similarity_percentage": round(similarity_score * 100, 2),
            "total_features": len(feature_names),
            "doc1_length": len(text1),
            "doc2_length": len(text2),
            "top_keywords_doc1": top_keywords_doc1,
            "top_keywords_doc2": top_keywords_doc2
        }
        
        if show_details:
            print("="*60)
            print("📊 HASIL ANALISIS SIMILARITY")
            print("="*60)
            print(f"\n🎯 Similarity Score: {result['similarity_percentage']}%")
            print(f"\n📏 Interpretasi:")
            if similarity_score >= 0.8:
                print("   ✅ Sangat Mirip (>80%)")
            elif similarity_score >= 0.6:
                print("   ⚠️  Cukup Mirip (60-80%)")
            elif similarity_score >= 0.4:
                print("   ⚡ Agak Mirip (40-60%)")
            else:
                print("   ❌ Tidak Mirip (<40%)")
            
            print(f"\n📝 Detail:")
            print(f"   - Dokumen 1: {result['doc1_length']:,} karakter")
            print(f"   - Dokumen 2: {result['doc2_length']:,} karakter")
            print(f"   - Total fitur TF-IDF: {result['total_features']:,}")
            
            print(f"\n🔑 Top Keywords Dokumen 1: {', '.join(top_keywords_doc1[:5])}")
            print(f"🔑 Top Keywords Dokumen 2: {', '.join(top_keywords_doc2[:5])}")
            print("="*60)
        
        return result
    
    except Exception as e:
        return {"error": str(e)}

In [0]:
# ==========================================
# 📝 INSTRUKSI CEPAT
# ==========================================
"""
STEP 1: ✅ File paths sudah di-detect! (lihat cell di atas)
STEP 2: Uncomment salah satu contoh di bawah sesuai kebutuhan
STEP 3: Run cell ini!
"""

print("\n" + "="*70)
print("📂 FILES YANG TERSEDIA DI VOLUMES:")
print("="*70)
print("\n📄 DOCX Files:")
print("   - Dokumen 1.docx (9.0 KB)")
print("   - Dokumen 2.docx (9.0 KB)")
print("   - Dokumen 3.docx (9.0 KB)")
print("   - Dokumen 4.docx (9.0 KB)")
print("\n📄 PDF Files:")
print("   - Dokumen 1.pdf (90.7 KB)")
print("   - Dokumen 4.pdf (91.7 KB)")
print("\n🎁 BONUS Files (also detected):")
print("   - file1.csv, file2.csv")
print("   - picture 1.jpg, picture 2.jpg")
print("\n" + "="*70)

# ==========================================
# ✅ BASE PATH (SUDAH CORRECT!)
# ==========================================
BASE_PATH = "/Volumes/workspace/bronze/volume"

print(f"\n📁 Base path: {BASE_PATH}")
print("✅ Path sudah correct! Siap digunakan!\n")


# ==========================================
# 🚀 CONTOH 1: Compare 2 DOCX Files
# ==========================================
print("\n🔵 CONTOH 1: Compare Dokumen 1.docx vs Dokumen 2.docx")
print("Uncomment 4 baris di bawah untuk menjalankan:\n")

file1 = f"{BASE_PATH}/Dokumen 1.docx"
file2 = f"{BASE_PATH}/Dokumen 2.docx"
text1 = extract_text_from_docx(file1)
text2 = extract_text_from_docx(file2)
result = calculate_similarity(text1, text2)


# ==========================================
# 🚀 CONTOH 2: Compare 2 PDF Files
# ==========================================
print("\n🔴 CONTOH 2: Compare Dokumen 1.pdf vs Dokumen 4.pdf")
print("Uncomment 4 baris di bawah untuk menjalankan:\n")

file1 = f"{BASE_PATH}/Dokumen 1.pdf"
file2 = f"{BASE_PATH}/Dokumen 4.pdf"
text1 = extract_text_from_pdf(file1)
text2 = extract_text_from_pdf(file2)
result = calculate_similarity(text1, text2)


# ==========================================
# 🚀 CONTOH 3: Compare PDF vs DOCX (Same Doc)
# ==========================================
print("\n🟡 CONTOH 3: Compare Dokumen 1.pdf vs Dokumen 1.docx")
print("(Check apakah PDF dan DOCX version identical)")
print("Uncomment 4 baris di bawah untuk menjalankan:\n")

file_pdf = f"{BASE_PATH}/Dokumen 1.pdf"
file_docx = f"{BASE_PATH}/Dokumen 1.docx"
text1 = extract_text_from_pdf(file_pdf)
text2 = extract_text_from_docx(file_docx)
result = calculate_similarity(text1, text2)


# ==========================================
# 🚀 CONTOH 4: Compare Dokumen 4 (PDF vs DOCX)
# ==========================================
print("\n🟢 CONTOH 4: Compare Dokumen 4.pdf vs Dokumen 4.docx")
print("Uncomment 4 baris di bawah untuk menjalankan:\n")

# file_pdf = f"{BASE_PATH}/Dokumen 4.pdf"
# file_docx = f"{BASE_PATH}/Dokumen 4.docx"
# text1 = extract_text_from_pdf(file_pdf)
# text2 = extract_text_from_docx(file_docx)
# result = calculate_similarity(text1, text2)


# ==========================================
# 🚀 CONTOH 5: Batch Compare All DOCX Files
# ==========================================
print("\n🟣 CONTOH 5: Batch compare semua kombinasi DOCX files")
print("(Total 6 comparisons: 1vs2, 1vs3, 1vs4, 2vs3, 2vs4, 3vs4)")
print("Uncomment code di bawah untuk batch comparison:\n")

import pandas as pd

docx_files = ["Dokumen 1.docx", "Dokumen 2.docx", "Dokumen 3.docx", "Dokumen 4.docx"]
results = []

print("🔄 Processing batch comparison...\n")
for i, file1_name in enumerate(docx_files):
    for file2_name in docx_files[i+1:]:
        file1 = f"{BASE_PATH}/{file1_name}"
        file2 = f"{BASE_PATH}/{file2_name}"
        
        print(f"🔍 Comparing: {file1_name} vs {file2_name}")
        text1 = extract_text_from_docx(file1)
        text2 = extract_text_from_docx(file2)
        result = calculate_similarity(text1, text2, show_details=False)
        
        results.append({
            "File 1": file1_name,
            "File 2": file2_name,
            "Similarity %": result['similarity_percentage']
        })
        print(f"   ✅ Similarity: {result['similarity_percentage']}%\n")

df_results = pd.DataFrame(results).sort_values('Similarity %', ascending=False)
print("\n" + "="*60)
print("📊 BATCH COMPARISON RESULTS:")
print("="*60)
display(df_results)


# ==========================================
# 🚀 CONTOH 6: Quick Test (No Files Needed)
# ==========================================
print("\n🟠 CONTOH 6: Quick test dengan sample text (no file upload needed)")
print("Uncomment 3 baris di bawah untuk test:\n")

# sample1 = "Databricks adalah platform data lakehouse untuk analytics dan machine learning"
# sample2 = "Databricks merupakan unified platform untuk data analytics dan ML workflows"
# result = calculate_similarity(sample1, sample2)


print("\n" + "="*70)
print("✅ Ready to Run! Pilih dan uncomment salah satu contoh di atas.")
print("="*70)
print("\n💡 CARA PAKAI:")
print("   1. Pilih contoh (1-6) yang sesuai kebutuhan")
print("   2. Hapus tanda # di awal baris (uncomment)")
print("   3. Run cell ini (Shift + Enter)")
print("   4. Lihat hasil similarity percentage!")
print("\n🎯 REKOMENDASI:")
print("   - Test dulu dengan CONTOH 6 (no files, cepat!)")
print("   - Lalu coba CONTOH 1 atau 2 dengan file Anda")
print("   - CONTOH 5 untuk batch comparison semua file\n")

#  jpeg & csv compare

In [0]:
# ==========================================
# 📸 Extract Text dari JPEG/PNG (OCR)
# ==========================================

# Install OCR library
%pip install pytesseract Pillow --quiet

from PIL import Image
import pytesseract

def extract_text_from_image(image_path, language='eng'):
    """
    Extract text dari image menggunakan OCR
    
    Args:
        image_path: Path ke image file (JPEG, PNG, etc)
        language: 'eng' (English), 'ind' (Indonesian)
    
    Returns:
        String berisi text hasil OCR
    """
    try:
        img = Image.open(image_path)
        
        print(f"📸 Image size: {img.size}")
        print(f"🔍 Performing OCR (this may take 5-10 seconds)...")
        
        # Perform OCR
        text = pytesseract.image_to_string(img, lang=language)
        
        print(f"✅ OCR completed!")
        print(f"📊 Total karakter: {len(text):,}")
        return text.strip()
    
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

print("✅ Function 'extract_text_from_image' ready!")

In [0]:
# ==========================================
# 📊 Compare CSV Files
# ==========================================

import pandas as pd
import numpy as np

def compare_csv_files(csv1_path, csv2_path, method='structure'):
    """
    Compare 2 CSV files
    
    Args:
        csv1_path: Path ke CSV pertama
        csv2_path: Path ke CSV kedua
        method: 'structure' (compare columns) atau 'content' (compare data)
    
    Returns:
        Dictionary berisi hasil comparison
    """
    try:
        # Read CSVs
        df1 = pd.read_csv(csv1_path)
        df2 = pd.read_csv(csv2_path)
        
        print(f"📊 CSV 1: {df1.shape[0]:,} rows × {df1.shape[1]} columns")
        print(f"📊 CSV 2: {df2.shape[0]:,} rows × {df2.shape[1]} columns\n")
        
        if method == 'structure':
            # Compare column structure
            cols1 = set(df1.columns)
            cols2 = set(df2.columns)
            
            common_cols = cols1.intersection(cols2)
            only_in_csv1 = cols1 - cols2
            only_in_csv2 = cols2 - cols1
            
            similarity = len(common_cols) / len(cols1.union(cols2)) * 100
            
            print("="*60)
            print("📊 STRUCTURE COMPARISON")
            print("="*60)
            print(f"\n🎯 Structure Similarity: {similarity:.2f}%")
            print(f"\n✅ Common Columns ({len(common_cols)}): {list(common_cols)[:5]}")
            if only_in_csv1:
                print(f"⚠️  Only in CSV 1: {list(only_in_csv1)}")
            if only_in_csv2:
                print(f"⚠️  Only in CSV 2: {list(only_in_csv2)}")
            print("="*60)
            
            return {
                "method": "structure",
                "similarity_percentage": round(similarity, 2),
                "common_columns": list(common_cols),
                "only_in_csv1": list(only_in_csv1),
                "only_in_csv2": list(only_in_csv2)
            }
        
        elif method == 'content':
            # Compare content using TF-IDF
            text1 = df1.to_string()
            text2 = df2.to_string()
            
            result = calculate_similarity(text1, text2, show_details=True)
            result['method'] = 'content'
            return result
            
    except Exception as e:
        print(f"❌ Error: {e}")
        return {"error": str(e)}

print("✅ Function 'compare_csv_files' ready!")

In [0]:
# ==========================================
# 🚀 EXAMPLES: Compare JPEG & CSV Files
# ==========================================

BASE_PATH = "/Volumes/workspace/bronze/volume"

print("="*70)
print("🎯 READY TO COMPARE JPEG & CSV FILES!")
print("="*70)

# ==========================================
# 📸 EXAMPLE 1: Compare 2 JPEG Images (OCR)
# ==========================================
print("\n📸 EXAMPLE 1: Compare picture 1.jpg vs picture 2.jpg")
print("Uncomment 5 baris di bawah:\n")

img1 = f"{BASE_PATH}/picture 1.jpg"
img2 = f"{BASE_PATH}/picture 2.jpg"
text1 = extract_text_from_image(img1, language='eng')  # ganti 'ind' untuk Bahasa Indonesia
text2 = extract_text_from_image(img2, language='eng')
result = calculate_similarity(text1, text2)


# ==========================================
# 📊 EXAMPLE 2: Compare CSV Structure
# ==========================================
print("\n📊 EXAMPLE 2: Compare file1.csv vs file2.csv (STRUCTURE)")
print("Check apakah kolom-kolomnya sama")
print("Uncomment 3 baris di bawah:\n")

csv1 = f"{BASE_PATH}/file1.csv"
csv2 = f"{BASE_PATH}/file2.csv"
result = compare_csv_files(csv1, csv2, method='structure')


# ==========================================
# 📊 EXAMPLE 3: Compare CSV Content
# ==========================================
print("\n📊 EXAMPLE 3: Compare file1.csv vs file2.csv (CONTENT)")
print("Check similarity data values")
print("Uncomment 3 baris di bawah:\n")

csv1 = f"{BASE_PATH}/file1.csv"
csv2 = f"{BASE_PATH}/file2.csv"
result = compare_csv_files(csv1, csv2, method='content')


# ==========================================
# 🎨 EXAMPLE 4: Mix - Compare PDF vs Image
# ==========================================
print("\n🎨 EXAMPLE 4: Compare Dokumen 1.pdf vs picture 1.jpg")
print("Useful jika image adalah scan dari PDF")
print("Uncomment 4 baris di bawah:\n")

pdf_file = f"{BASE_PATH}/Dokumen 1.pdf"
img_file = f"{BASE_PATH}/picture 1.jpg"
text1 = extract_text_from_pdf(pdf_file)
text2 = extract_text_from_image(img_file)
result = calculate_similarity(text1, text2)


print("\n" + "="*70)
print("✅ All functions ready! Uncomment examples above to test.")
print("="*70)
print("\n⚠️  NOTE untuk IMAGE OCR:")
print("   - OCR takes 5-10 seconds per image (large files)")
print("   - Accuracy: 70-90% (depends on image quality)")
print("   - Works best for: Clear text, high resolution, good contrast")
print("\n💡 NOTE untuk CSV:")
print("   - 'structure' method: Compare columns only (fast)")
print("   - 'content' method: Compare data values (slower)")

# Compare CSV Detail

In [0]:
# ==========================================
# 📊 PROPER CSV COMPARISON (Advanced)
# ==========================================

import pandas as pd
import numpy as np

def compare_csv_advanced(csv1_path, csv2_path):
    """
    Advanced CSV comparison dengan multiple metrics
    """
    df1 = pd.read_csv(csv1_path)
    df2 = pd.read_csv(csv2_path)
    
    print("="*70)
    print("📊 ADVANCED CSV COMPARISON")
    print("="*70)
    
    # 1. BASIC INFO
    print(f"\n📋 Basic Info:")
    print(f"   CSV 1: {df1.shape[0]:,} rows × {df1.shape[1]} columns")
    print(f"   CSV 2: {df2.shape[0]:,} rows × {df2.shape[1]} columns")
    
    # 2. STRUCTURE COMPARISON
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    common_cols = cols1.intersection(cols2)
    
    structure_similarity = len(common_cols) / len(cols1.union(cols2)) * 100
    print(f"\n🏗️  Structure Similarity: {structure_similarity:.2f}%")
    print(f"   Common columns: {len(common_cols)}")
    
    # 3. ROW COUNT COMPARISON
    row_diff = abs(df1.shape[0] - df2.shape[0])
    row_diff_pct = (row_diff / max(df1.shape[0], df2.shape[0])) * 100
    print(f"\n📏 Row Count Difference:")
    print(f"   Difference: {row_diff} rows ({row_diff_pct:.1f}%)")
    if row_diff == 0:
        print(f"   ✅ Same number of rows")
    elif row_diff_pct < 10:
        print(f"   ⚠️  Minor difference")
    else:
        print(f"   ❌ Significant difference")
    
    # 4. DATA OVERLAP (for common columns)
    if len(common_cols) > 0:
        print(f"\n🔄 Data Overlap Analysis:")
        
        # Check unique values overlap for categorical columns
        overlap_scores = []
        for col in list(common_cols)[:5]:  # Top 5 columns
            try:
                unique1 = set(df1[col].dropna().astype(str))
                unique2 = set(df2[col].dropna().astype(str))
                
                if len(unique1) > 0 or len(unique2) > 0:
                    overlap = len(unique1.intersection(unique2))
                    total_unique = len(unique1.union(unique2))
                    overlap_pct = (overlap / total_unique * 100) if total_unique > 0 else 0
                    overlap_scores.append(overlap_pct)
                    print(f"   {col}: {overlap_pct:.1f}% value overlap")
            except:
                pass
        
        if overlap_scores:
            avg_overlap = np.mean(overlap_scores)
            print(f"\n   📊 Average Data Overlap: {avg_overlap:.1f}%")
    
    # 5. OVERALL SIMILARITY SCORE
    # Weighted average: 40% structure, 30% row count, 30% data overlap
    row_similarity = 100 - row_diff_pct
    data_similarity = np.mean(overlap_scores) if overlap_scores else 0
    
    overall = (structure_similarity * 0.4) + (row_similarity * 0.3) + (data_similarity * 0.3)
    
    print(f"\n" + "="*70)
    print(f"🎯 OVERALL CSV SIMILARITY: {overall:.2f}%")
    print("="*70)
    
    print(f"\n📊 Breakdown:")
    print(f"   - Structure (40%): {structure_similarity:.1f}%")
    print(f"   - Row Count (30%): {row_similarity:.1f}%")
    print(f"   - Data Overlap (30%): {data_similarity:.1f}%")
    
    # Interpretation
    print(f"\n📏 Interpretation:")
    if overall >= 90:
        print("   ✅ CSV files are ALMOST IDENTICAL")
    elif overall >= 70:
        print("   ⚠️  CSV files are SIMILAR but have differences")
    elif overall >= 50:
        print("   ⚡ CSV files are SOMEWHAT RELATED")
    else:
        print("   ❌ CSV files are DIFFERENT")
    
    return {
        "overall_similarity": round(overall, 2),
        "structure_similarity": round(structure_similarity, 2),
        "row_similarity": round(row_similarity, 2),
        "data_overlap": round(data_similarity, 2),
        "row_diff": row_diff,
        "row_diff_pct": round(row_diff_pct, 2)
    }

print("✅ Function 'compare_csv_advanced' ready!")

In [0]:
# Test advanced CSV comparison
csv1 = "/Volumes/workspace/bronze/volume/file1.csv"
csv2 = "/Volumes/workspace/bronze/volume/file2.csv"

result = compare_csv_advanced(csv1, csv2)